# Workflow 4: Population-Level Detection of Misfolding Involving Native Entanglements

## Goal

Use proteome-scale structures and LiP-MS-derived labels to test enrichment of conformational signals in native entangled regions and identify high-risk protein subpopulations.

This notebook follows the schema of Workflows 1-3 and mirrors [workflow4_population.md](workflow4_population.md).

### Runtime notes
- Native NCLE batch over the full tutorial cohort can take hours to days.
- This notebook is configured to run the full tutorial cohort (no top-20 subselect).
- Regression and Monte Carlo run directly from the full-cohort outputs.

> Environment: activate `ncledetector` and select that kernel before running cells.

In [ ]:
import os,sys
import subprocess
from pathlib import Path
import pandas as pd

DATASTORE = "/scratch/ims86/NCLEdetector_Datastore"
REPO = "/storage/group/epo2/default/ims86/git_repos/NCLEdetector"
STRUCTDIR = f"{DATASTORE}/user_input/proteome_structures"
OUTDIR = f"{DATASTORE}/outputs/workflow4"
GENELIST_AF = f"{DATASTORE}/user_input/experimental_data/Gene_lists/AF/AF_0.6g_C_Rall_spa50_LiPMScov50_ent_genes.txt"
RES_FEATURES_AF = f"{DATASTORE}/user_input/experimental_data/PDB_residue_features/AF/residueFeatures.csv"

NATIVE_BATCH_OUTDIR = f"{OUTDIR}/nativeNCLE_all"
DESIGN_MATRIX_FILE = f"{OUTDIR}/residue_dataframes_workflow4.csv"
REG_OUTDIR = f"{OUTDIR}/population_modeling"
MC_OUTDIR = f"{OUTDIR}/monte_carlo"
LOGDIR = f"{OUTDIR}/logs"

for p in [NATIVE_BATCH_OUTDIR, REG_OUTDIR, MC_OUTDIR, LOGDIR]:
    os.makedirs(p, exist_ok=True)

print("DATASTORE:", DATASTORE)
print("STRUCTDIR:", STRUCTDIR)
print("OUTDIR:", OUTDIR)
print("GENELIST_AF:", GENELIST_AF)
print("RES_FEATURES_AF:", RES_FEATURES_AF)
print("\nOutput directories ready.")

## Step 1. Activate your environment and set paths

The next cells define the tutorial paths, create the output directories, and verify that the required AF structures and gene-list inputs are present.

In [ ]:
af_dir = Path(f"{STRUCTDIR}/AF")
exp_dir = Path(f"{STRUCTDIR}/EXP")
gene_file = Path(GENELIST_AF)

assert af_dir.is_dir(), f"Missing AF directory: {af_dir}"
assert exp_dir.is_dir(), f"Missing EXP directory: {exp_dir}"
assert gene_file.is_file(), f"Missing gene list: {gene_file}"

af_pdbs = sorted(af_dir.glob("*.pdb"))
print(f"AF PDB count: {len(af_pdbs)}")
print("AF sample:", [p.name for p in af_pdbs[:5]])

with gene_file.open("r", encoding="utf-8") as fh:
    genes_all = [ln.strip() for ln in fh if ln.strip() and not ln.startswith("#")]

print(f"Gene list entries: {len(genes_all)}")
print("Gene sample:", genes_all[:10])

### Tutorial cohort selection

Use the full tutorial AF ent-gene list for batch NCLE and downstream modeling.

In [ ]:
genes_full = genes_all
print(f"Using full tutorial gene list with {len(genes_full)} genes")
print("Gene sample:", genes_full[:10])

## Step 2. Run Workflow 1 in batch to generate native NCLEs per protein

Run the single-script wrapper `run_workflow4_nativeNCLE_batch.py` in two stages:
1. Dry run to inspect the selected proteins
2. Actual parallel run on the full tutorial cohort

This batch step also builds the combined residue-level design matrix used by the regression and Monte Carlo steps.

In [ ]:
import shlex

dry_cmd = [
    "python", "scripts/run_workflow4_nativeNCLE_batch.py",
    "--pdb_dir", f"{STRUCTDIR}/AF",
    "--gene_list", GENELIST_AF,
    "--outdir", NATIVE_BATCH_OUTDIR,
    "--organism", "Ecoli",
    "--model", "AF",
    "--ent_detection_method", "3",
    "--g_threshold", "0.6",
    "--density", "1.0",
    "--residue_features_file", RES_FEATURES_AF,
    "--reg_formula", "cut_C_Rall ~ AA + region",
    "--design_matrix_file", DESIGN_MATRIX_FILE,
    "--nproc", "8",
    "--logdir", LOGDIR,
    "--dry_run",
]

print("Running dry-run command:")
print(shlex.join(dry_cmd))

proc = subprocess.run(
    dry_cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
 )
print(proc.stdout[-4000:])
print("Return code:", proc.returncode)


In [ ]:
run_cmd = [
    "python", "scripts/run_workflow4_nativeNCLE_batch.py",
    "--pdb_dir", f"{STRUCTDIR}/AF",
    "--gene_list", GENELIST_AF,
    "--outdir", NATIVE_BATCH_OUTDIR,
    "--organism", "Ecoli",
    "--model", "AF",
    "--ent_detection_method", "3",
    "--g_threshold", "0.6",
    "--density", "1.0",
    "--residue_features_file", RES_FEATURES_AF,
    "--reg_formula", "cut_C_Rall ~ AA + region",
    "--design_matrix_file", DESIGN_MATRIX_FILE,
    "--nproc", "8",
    "--logdir", LOGDIR,
]

print("Running batch nativeNCLE for full tutorial cohort...")
print(shlex.join(run_cmd))

proc = subprocess.run(
    run_cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
 )
print(proc.stdout[-4000:])
print("Return code:", proc.returncode)


In [ ]:
out_root = Path(NATIVE_BATCH_OUTDIR)
protein_dirs = sorted([p for p in out_root.iterdir() if p.is_dir()])
print(f"Per-protein output dirs found: {len(protein_dirs)}")
print([p.name for p in protein_dirs[:10]])

# Quick output integrity check
required_subdirs = [
    "Native_GE",
    "Native_HQ_GE",
    "Native_clustered_HQ_GE",
    "Native_clustered_HQ_GE_features",
]

if protein_dirs:
    sample = protein_dirs[0]
    print(f"\nSample protein dir: {sample}")
    for sd in required_subdirs:
        print(sd, (sample / sd).exists())

### Validate combined design matrix

Step 2 writes a single combined matrix to `DESIGN_MATRIX_FILE` with required columns:
`gene`, `mapped_resid`, `uniprot_length`, `AA`, `region`, `cut_C_Rall`.

The next cell verifies file presence and previews the schema.

In [ ]:
design_matrix = Path(DESIGN_MATRIX_FILE)
assert design_matrix.is_file(), f"Missing design matrix: {design_matrix}"

print(f"Design matrix file: {design_matrix}")
print(f"Size (MB): {design_matrix.stat().st_size / (1024**2):.2f}")

df0 = pd.read_csv(design_matrix, sep="|", nrows=5)
print("Sample columns:", list(df0.columns))
print(df0)

## Step 3. Test proteome-level enrichment of experimental signal in native entangled regions

In [ ]:
reg_cmd = [
    "python", "scripts/run_population_modeling.py",
    "--dataframe_files", DESIGN_MATRIX_FILE,
    "--outdir", REG_OUTDIR,
    "--gene_list", GENELIST_AF,
    "--ID", "Ecoli_population",
    "--reg_formula", "cut_C_Rall ~ AA + region",
    "--logdir", LOGDIR,
]

print("Running proteome regression...")
print(shlex.join(reg_cmd))

proc = subprocess.run(
    reg_cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
 )
print(proc.stdout[-4000:])
print("Return code:", proc.returncode)


## Step 4. Monte Carlo subpopulation selection (full tutorial cohort)

This cell uses tutorial-scale settings (`steps=100000`).

In [ ]:
mc_cmd = [
    "python", "scripts/run_montecarlo.py",
    "--dataframe_files", DESIGN_MATRIX_FILE,
    "--outdir", MC_OUTDIR,
    "--gene_list", GENELIST_AF,
    "--ID", "Ecoli_population_mc",
    "--reg_formula", "cut_C_Rall ~ region + AA",
    "--response_var", "cut_C_Rall",
    "--test_var", "region",
    "--n_groups", "4",
    "--steps", "100000",
    "--C1", "1.0",
    "--C2", "2.5",
    "--beta", "0.05",
    "--logdir", LOGDIR,
]

print("Running Monte Carlo...")
print(shlex.join(mc_cmd))

proc = subprocess.run(
    mc_cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
 )
print(proc.stdout[-4000:])
print("Return code:", proc.returncode)


In [ ]:
mc_out = Path(MC_OUTDIR)
final_files = sorted(mc_out.glob("Final_step_reg_*"))
state_gene_files = sorted(mc_out.glob("State*_final_genelist_*"))
state_traj_files = sorted(mc_out.glob("State*_final_traj_*"))

print("Monte Carlo output summary:")
print("  Final reg files:", len(final_files))
print("  Final genelist files:", len(state_gene_files))
print("  Final trajectory files:", len(state_traj_files))

print("\nSample outputs:")
for p in (final_files[:2] + state_gene_files[:2] + state_traj_files[:2]):
    print(" ", p.name)

## Step 5. Next actions

- Aggregate repeated Monte Carlo runs and rank proteins by inclusion frequency in top-enrichment states.
- Compare top-state protein sets against independent biological annotations.
- Tune `n_groups`, `C1`, `C2`, and `beta` based on your desired tradeoff between enrichment and size-distribution matching.